# 옥트리 생성 & 옥트리 기반 CCL 시각화

CloudCompare의 *Label Connected Components* 는 클러스터링이 아니라 **옥트리(복셀) 기반 CCL(Connected Component Labeling)** 입니다. 이 노트북은:

1. **옥트리 생성 과정**을 깊이별로 시각화해 이해하고
2. 특정 옥트리 레벨 = **균일 복셀 그리드**임을 확인한 뒤
3. 점유 복셀을 **연결요소로 묶어(CCL)** 주변 노이즈를 제거합니다.

## 왜 CCL인가 (DBSCAN 대신)

포인트가 250만 개 수준이면 DBSCAN은 이웃 거리 계산에서 메모리가 터집니다. 옥트리/복셀 CCL은 점을 복셀에 담고 인접 복셀만 BFS로 묶으므로 **O(N)** 으로 가볍고 메모리 안전합니다.

> 시각화는 전체 점이 아니라 **다운샘플**해서 그립니다 (plotly가 무거워지지 않게).
> 핵심 함수는 모두 [`util.pcd_tool`](../../python/util/pcd_tool.py)에 있습니다.

In [14]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[1] / 'python'))  # util.pcd_tool 경로

import numpy as np
import open3d as o3d

from util.pcd_tool import (
    load_pcd,
    build_octree, octree_boxes_lineset,
    voxel_ccl, colorize_ccl,
    save_pcd, compare_with_reference,
    make_pcd, show3d, downsample_for_view, RGB,
)

np.random.seed(42)

## 1. 로드

**처리는 전체 해상도** PCD(`pcd`)로 수행합니다. 옥트리/복셀 CCL은 O(N)이라 수백만 점도 메모리 안전합니다. (단, 파이썬 루프라 수백만 점이면 CCL이 수초~십수초 걸릴 수 있습니다.)

> **다운샘플은 시각화(plotly 렌더링)에만** 사용합니다 — `downsample_for_view()`. 처리 데이터는 다운샘플하지 않습니다.
> SOR(통계적 이상치 제거)은 쓰지 않고 CCL 단독으로 테스트합니다.

In [15]:
# ====== 설정 ======
PCD_PATH = "../../sample/MERGED_SPOOL-004_0001_20260528T170812.pcd"
SCALE    = 1e-3    # mm→m
# ==================

# 처리에 쓸 전체 해상도 PCD (다운샘플 X)
pcd = load_pcd(PCD_PATH, scale=SCALE)
print(f"로드: {len(pcd.points):,} 점 (전체 해상도)")
print(f"범위: {np.asarray(pcd.points).min(0).round(3)} ~ {np.asarray(pcd.points).max(0).round(3)} m")

# 시각화는 다운샘플본으로만
show3d([make_pcd(downsample_for_view(pcd), 'gray')])
print('1. 원본 PCD (시각화는 다운샘플)')

로드: 2,600,209 점 (전체 해상도)
범위: [-0.116  0.959 -0.038] ~ [5.409 2.088 1.369] m


1. 원본 PCD (시각화는 다운샘플)


## 2. 옥트리 생성 과정 시각화

`build_octree(pcd, max_depth)` 로 옥트리를 만들고, `octree_boxes_lineset(octree, depth)` 로 **각 깊이의 노드 큐브**를 와이어프레임으로 그립니다.

옥트리는 전체 바운딩 큐브를 8등분해 가며 내려갑니다:
- depth 0 = 전체 큐브 1개
- depth d = 점이 있는 영역만 $8^d$ 까지 세분 (빈 공간은 분할 안 함)

깊이가 깊어질수록 셀이 작아지고 객체 표면을 촘촘히 감쌉니다. 아래에서 깊이를 바꿔가며 확인하세요.

In [28]:
OCTREE_MAX_DEPTH = 8
octree = build_octree(pcd, max_depth=OCTREE_MAX_DEPTH)
print(f"옥트리 생성: max_depth={OCTREE_MAX_DEPTH}, origin={np.round(octree.origin,3)}, size={octree.size:.3f} m")

# 깊이별 노드 개수
for d in range(OCTREE_MAX_DEPTH + 1):
    _, n = octree_boxes_lineset(octree, depth=d)
    cell = octree.size / (2 ** d)
    print(f"  depth {d}: 노드 {n:5d}개,  셀 크기 {cell:.4f} m")

옥트리 생성: max_depth=8, origin=[-0.116 -1.239 -2.097], size=5.580 m
  depth 0: 노드     1개,  셀 크기 5.5803 m
  depth 1: 노드     8개,  셀 크기 2.7902 m
  depth 2: 노드    14개,  셀 크기 1.3951 m
  depth 3: 노드    24개,  셀 크기 0.6975 m
  depth 4: 노드    45개,  셀 크기 0.3488 m
  depth 5: 노드    84개,  셀 크기 0.1744 m
  depth 6: 노드   339개,  셀 크기 0.0872 m
  depth 7: 노드  1205개,  셀 크기 0.0436 m
  depth 8: 노드  4828개,  셀 크기 0.0218 m


## 3. 옥트리 레벨 선택

CloudCompare CCL은 **옥트리 레벨**을 매개변수로 받습니다. 레벨 `L`을 고르면 셀 크기는 자동으로 `octree.size / 2**L` 로 정해집니다 (위 2번 표 참고).

여기서도 복셀 크기(m)를 직접 쓰지 않고 **`OCTREE_LEVEL` 하나만** 지정합니다. 분리하려는 객체 간격에 맞는 레벨을 위 표에서 골라 아래에 넣으세요. (레벨↑ = 셀 작아짐 = 분리 잘 되지만 과분할 위험)

In [30]:
# ====== 옥트리 레벨만 지정 ======
OCTREE_LEVEL = 8   # 이 레벨의 셀 크기로 CCL 수행 (복셀 크기는 자동 계산)
CCL_VOXEL = octree.size / (2 ** OCTREE_LEVEL)
# ================================

print(f"옥트리 레벨 {OCTREE_LEVEL}  →  복셀 크기 {CCL_VOXEL:.4f} m")

# 해당 레벨의 점유 복셀 그리드 시각화 (전체 해상도 기준)
vg = o3d.geometry.VoxelGrid.create_from_point_cloud(pcd, voxel_size=CCL_VOXEL)
print(f"점유 복셀 수: {len(vg.get_voxels()):,}")
print(f'3. CCL이 사용할 복셀 그리드 (옥트리 레벨 {OCTREE_LEVEL})')

옥트리 레벨 8  →  복셀 크기 0.0218 m
점유 복셀 수: 5,053
3. CCL이 사용할 복셀 그리드 (옥트리 레벨 8)


## 4. 옥트리(복셀) 기반 CCL

`voxel_ccl(points, voxel_size, min_points, connectivity)` — 점유 복셀을 26-이웃 BFS로 연결요소(component)로 묶습니다. 떨어져 있는 노이즈 덩어리는 별도 컴포넌트가 됩니다.

- `voxel_size`: 위에서 고른 **옥트리 레벨**로부터 자동 계산된 값(`CCL_VOXEL`)
- `min_points`: 이보다 작은 컴포넌트는 버림
- `connectivity`: 6(면) 또는 26(면+모서리+꼭짓점)

In [ ]:
pts = np.asarray(pcd.points)
kept_idx, labels = voxel_ccl(pts, voxel_size=CCL_VOXEL, min_points=30, connectivity=26)

n_comp = labels.max() + 1
uniq, cnts = np.unique(labels[labels >= 0], return_counts=True)
print(f"컴포넌트 수: {n_comp}  (버려진 점: {(labels < 0).sum():,})")
for c, n in sorted(zip(uniq, cnts), key=lambda x: -x[1])[:10]:
    print(f"  component {c}: {n:,} 점")


    

# 컴포넌트별 색칠 (전체 색칠 후 시각화용으로만 다운샘플 → 색-점 대응 유지)
colored = colorize_ccl(pcd, labels)
show3d([downsample_for_view(colored, 30000)])
print('4. CCL 컴포넌트별 색상 (검정 = 버려진 점)')

컴포넌트 수: 10  (버려진 점: 138)
  component 0: 2,598,470 점
  component 5: 581 점
  component 1: 406 점
  component 4: 152 점
  component 7: 116 점
  component 8: 87 점
  component 6: 74 점
  component 2: 72 점
  component 9: 69 점
  component 3: 44 점


4. CCL 컴포넌트별 색상 (검정 = 버려진 점)


## 5. 노이즈 제거 결과

`denoise_voxel_ccl(pcd, voxel_size, min_points, keep='largest')` — 가장 큰 컴포넌트(주 객체)만 남겨 주변 노이즈를 제거합니다.

In [ ]:




print(f"정제 전 {len(pcd.points):,} → 정제 후 {len(pcd_clean.points):,} 점")

show3d([make_pcd(downsample_for_view(pcd_clean, 30000), 'blue')])
print('5. CCL 노이즈 제거 완료 (가장 큰 컴포넌트만)')

정제 전 2,600,209 → 정제 후 2,598,470 점


5. CCL 노이즈 제거 완료 (가장 큰 컴포넌트만)


## 요약 & 재사용

```python
from util.pcd_tool import load_pcd, voxel_ccl, save_pcd

pcd = load_pcd("spool.pcd", scale=1e-3)
# 옥트리 레벨 → 복셀 크기 (CloudCompare CCL과 동일한 감각)
octree = build_octree(pcd.voxel_down_sample(0.01), max_depth=8)
voxel = pcd.get_axis_aligned_bounding_box().get_extent().max() / 2**7
# 대용량도 메모리 안전 (DBSCAN 대체)
kept, labels = voxel_ccl(np.asarray(pcd.points), voxel, min_points=30)
```

| 단계 | 함수 | 핵심 파라미터 |
|------|------|----------------|
| 옥트리 생성 | `build_octree` | `max_depth` |
| 옥트리 셀 시각화 | `octree_boxes_lineset` | `depth` |
| 레벨→복셀 크기 | 복셀 크기 계산 | `level` |
| CCL 라벨링 | `voxel_ccl` | `voxel_size`, `min_points`, `connectivity` |
| CCL 노이즈 제거 | `voxel_ccl` | `voxel_size`, `min_points`, `keep` |
| 저장 | `save_pcd` | `path` |
| 상용툴 비교 | `compare_with_reference` | `icp_threshold`, `ref_scale` |

**팁:** 이제 복셀 크기(m)를 직접 외우지 않고 **옥트리 레벨**만 고르면 됩니다. 레벨이 높을수록 셀이 작아져 분리가 잘 되지만, 너무 높으면 표면이 끊겨 한 객체가 여러 조각으로 과분할됩니다. 2번 셀의 깊이별 셀 크기 표를 보고 분리하려는 객체 간격에 맞는 레벨을 고르세요.

In [25]:
# ====== 저장 설정 ======
import pathlib
OUT_PATH = pathlib.Path(PCD_PATH).with_name(
    pathlib.Path(PCD_PATH).stem + "_ccl_clean.pcd")
# =======================

saved = save_pcd(pcd_clean, OUT_PATH)
print(f"저장 완료: {saved}")
print(f"  {len(pcd_clean.points):,} 점")

저장 완료: ../../sample/MERGED_SPOOL-004_0001_20260528T170812_ccl_clean.pcd
  2,598,470 점


## 7. 상용툴 결과와 비교 (ICP 정합 후)

상용툴(예: CloudCompare)에서 필터링한 결과 PCD와 내 CCL 결과를 비교합니다.

좌표계/원점이 다를 수 있으므로 **ICP로 먼저 정합**한 뒤:
- **점 개수 차이** — 두 필터가 남긴 점 수가 얼마나 다른가
- **평균 최근접 점거리** — 정합 후 한쪽 점에서 다른 쪽 가장 가까운 점까지 거리의 평균 (양방향 + 대칭)

을 계산합니다. 대칭 평균 거리가 작을수록 두 결과의 형상이 일치합니다.

`compare_with_reference(result, reference, icp_threshold, ref_scale)` 사용. 상용툴 결과가 mm 단위면 `ref_scale=1e-3`."

In [26]:
# ====== 비교 설정 ======
REF_PATH      = "../../sample/MERGED_SPOOL-004_0001_20260528T170812_filtered.pcd"  # 상용툴 필터 결과 경로
REF_SCALE     = 1.0     # 상용툴 결과 단위 (mm 단위면 1e-3)
ICP_THRESHOLD = 0.05    # ICP 대응점 최대 거리 (m)
# =======================

import os
if not os.path.exists(REF_PATH):
    print(f"[!] 비교 대상 파일이 없습니다: {REF_PATH}")
    print("    REF_PATH를 상용툴(CloudCompare 등) 필터 결과 파일로 바꾸세요.")
else:
    reference = load_pcd(REF_PATH, scale=1.0)  # ref 단위 변환은 compare 내부에서 적용
    cmp = compare_with_reference(pcd_clean, reference,
                                 icp_threshold=ICP_THRESHOLD, ref_scale=REF_SCALE)

    print("===== 상용툴 결과 vs 내 CCL 결과 =====")
    print(f"  ICP fitness / RMSE  : {cmp['icp_fitness']:.4f} / {cmp['icp_rmse']:.5f} m")
    print("  ---- 점 개수 ----")
    print(f"  내 결과 (result)    : {cmp['n_result']:,}")
    print(f"  상용툴 (reference)  : {cmp['n_reference']:,}")
    print(f"  개수 차이           : {cmp['count_diff']:+,} ({cmp['count_diff_ratio']*100:+.2f}%)")
    print("  ---- 평균 최근접 거리 ----")
    print(f"  result → reference  : {cmp['mean_dist_r2ref']:.5f} m")
    print(f"  reference → result  : {cmp['mean_dist_ref2r']:.5f} m")
    print(f"  대칭 평균           : {cmp['mean_dist_symmetric']:.5f} m")

    # 정합 결과 시각화: 내 결과(파랑) vs 상용툴(주황)
    ref_view = make_pcd(downsample_for_view(reference, 30000), 'orange')
    if REF_SCALE != 1.0:
        ref_view.scale(REF_SCALE, center=(0, 0, 0))
    show3d([
        make_pcd(downsample_for_view(cmp['aligned_result'], 30000), 'blue'),
        ref_view,
    ])
    print('7. ICP 정합 후 겹쳐보기 (파랑=내 결과, 주황=상용툴)')

===== 상용툴 결과 vs 내 CCL 결과 =====
  ICP fitness / RMSE  : 0.0000 / 0.00000 m
  ---- 점 개수 ----
  내 결과 (result)    : 2,598,470
  상용툴 (reference)  : 2,598,470
  개수 차이           : +0 (+0.00%)
  ---- 평균 최근접 거리 ----
  result → reference  : 1473.22507 m
  reference → result  : 3317.58585 m
  대칭 평균           : 2395.40546 m


7. ICP 정합 후 겹쳐보기 (파랑=내 결과, 주황=상용툴)


## 요약 & 재사용

```python
from util.pcd_tool import load_pcd, voxel_ccl

pcd = load_pcd("spool.pcd", scale=1e-3)
# 대용량도 메모리 안전 (DBSCAN 대체). 필요하면 먼저 voxel_down_sample 후 적용
kept, labels = voxel_ccl(np.asarray(pcd.points), 0.04, min_points=30)
```

| 단계 | 함수 | 핵심 파라미터 |
|------|------|----------------|
| 옥트리 생성 | `build_octree` | `max_depth` |
| 옥트리 셀 시각화 | `octree_boxes_lineset` | `depth` |
| CCL 라벨링 | `voxel_ccl` | `voxel_size`, `min_points`, `connectivity` |
| CCL 노이즈 제거 | `voxel_ccl` | `voxel_size`, `min_points`, `keep` |

**팁:** `voxel_size`(=옥트리 레벨)가 가장 중요합니다. 분리하려는 객체 간 최소 간격보다 작게 잡아야 노이즈 덩어리가 별도 컴포넌트로 떨어집니다. 너무 작으면 표면이 끊겨 한 객체가 여러 조각으로 과분할됩니다.